# 5c (Capstone) - Mining with the Two-Way Functionality Check

Notebook 5b mined the policy store through the `gmstest` wrapper (`CatalogBaseSource`), which is convenient for teaching but drives the `knowlytix.benchmark.generators` family: the questions are admitted by a **symbolic** recompute only, and its `multi_hop` generator emits a bare extremal ("which entry has the highest value"), which is really a *ranking* question.

Here we use the **production** answer-key family, `knowlytix.benchmark.eval.generators`, with `geometric=True`. Two things change, and both matter:

1. **The two-way check is the actual guard.** Each question is admitted only if the symbolic oracle *and* the trained GMS geometry recover its answer --- not demonstrated on the side, but run as the admission gate on every mined item, with an auditable recovery split.
2. **`multi_hop` is a genuine two-hop.** It composes an aggregation (argmin / argmax over a register) *and then* a relational lookup, and the second-hop triples are what the geometry recovers.

**What the next cell does** --- loads the capstone policy store and names the production generator family:

1. **Imports.** Pulls the six `knowlytix.benchmark.eval.generators` classes and the store loader.
2. **Find and load the store.** Walks up from the cwd to `data/gms_policy_store_geode`, loads it on CPU and asserts `store.load()`.
3. **List the family.** Prints each generator's own `category` label --- the categories whose answer key the capstone is built from.

In [ ]:
import torch, pathlib
from collections import Counter
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from knowlytix.benchmark.eval.generators.exact_recall import ExactRecallGenerator
from knowlytix.benchmark.eval.generators.counting import CountingGenerator
from knowlytix.benchmark.eval.generators.cross_reference import CrossReferenceGenerator
from knowlytix.benchmark.eval.generators.multi_hop import MultiHopGenerator
from knowlytix.benchmark.eval.generators.contradiction import ContradictionGenerator
from knowlytix.benchmark.eval.generators.threshold import ThresholdGenerator

ROOT = pathlib.Path.cwd().parents[1] / 'beyond-prompt-and-pray' / 'code'
while not (ROOT / 'data' / 'gms_policy_store_geode').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
store = GMSExpertStore(
    DocGMSConfig(store_path=str(ROOT / 'data' / 'gms_policy_store_geode'), ingest_mode='regex'),
    device=torch.device('cpu'))
assert store.load()

GEN_CLASSES = [ExactRecallGenerator, CountingGenerator, CrossReferenceGenerator,
               MultiHopGenerator, ContradictionGenerator, ThresholdGenerator]
print('eval generators (production answer-key family):', [C.category for C in GEN_CLASSES])

## Mine every category with the geometry as the guard
We run each generator with `geometric=True`. `_validate` then admits a question only when the symbolic recompute and the geometric recovery agree, and tallies *how* each was verified on `recovery_stats` --- `two_way` (symbolic + geometry), `symbolic` (an aggregation with no fact triple to recover, recused with a stated reason) or `rejected`. Nothing is admitted silently.

**What the next cell does** --- mines all six categories under the two-way check and prints the per-category admission and recovery split:

1. **Mine.** For each generator class, instantiate with `geometric=True`, run `generate(store)` and stash `(generator, questions)` in `mined`.
2. **Report.** Print, per category, how many questions were admitted and the `recovery_stats` split that admitted them.

In [ ]:
mined = {}
for Cls in GEN_CLASSES:
    g = Cls(geometric=True)
    qs = g.generate(store)
    mined[g.category] = (g, qs)

print(f"{'category':16s}{'admitted':>9s}   recovery split")
for cat, (g, qs) in mined.items():
    print(f'{cat:16s}{len(qs):9d}   {dict(g.recovery_stats)}')

## The recovery split is per-question and auditable
A `two_way` tag means the geometry recovered the answer's fact triple --- ranked the true tail first among the relation's non-asserted tails, and placed it under the relation's learned `cap_radius` (the calibrated operating point, read from the store, never hand-set). A `symbolic:<reason>` tag means the question is an aggregation (a count, a boolean) with no fact triple, so the symbolic recompute is the complete check and the recusal is recorded with its reason. The geometry never silently skips.

**What the next cell does** --- shows the per-question recovery tags and the calibrated cut for three representative categories:

1. **Tally tags.** For `exact_recall`, `counting` and `multi_hop`, count the `metadata['_recovery']` tag across the admitted questions.
2. **Show the cut.** Print each generator's `last_recovery_note` --- the learned `cap_radius` the geometric gate read (or `None` when every question recused to symbolic).

In [ ]:
for cat in ('exact_recall', 'counting', 'multi_hop'):
    g, qs = mined[cat]
    tags = Counter(q.metadata.get('_recovery') for q in qs)
    print(f'{cat}:')
    for tag, n in tags.items():
        print(f'    {n:3d}  {tag}')
    print(f'    calibrated cut: {g.last_recovery_note}')

## `multi_hop` is a genuine two-hop, and the geometry recovers the second hop
Unlike 5b's bare extremal, this `multi_hop` question composes two steps: **hop 1** finds the extremal entry of a register (a symbolic aggregation the geometry cannot verify), and **hop 2** traverses that entity's relation. The ground truth is the triple `(head, value, tails)`, and it is the *second-hop* relationship triple that the geometry recovers --- which is what earns the `two_way` tag.

**What the next cell does** --- pulls one two-way `multi_hop` question and unpacks its two hops:

1. **Pick.** Take the first `multi_hop` item tagged `two_way`.
2. **Unpack.** Split its `(head, value, tails)` ground truth and read the second-hop relation from `metadata['relation']`.
3. **Show both hops** and the recovery tag confirming the second-hop triple was geometrically recovered.

In [ ]:
g, qs = mined['multi_hop']
q = next(q for q in qs if q.metadata.get('_recovery') == 'two_way')
head, value, tails = q.ground_truth
print('Q:', q.natural_language)
print('hop 1 (aggregate):', head, '=', value,
      '  (extremal entry in', q.metadata['enm_type'] + ')')
print('hop 2 (traverse) :', f"{head} --{q.metadata['relation']}--> {sorted(tails)}")
print('recovery         :', q.metadata['_recovery'],
      '-> the second-hop triple is geometrically recovered')

## The geometry earns its keep: it rejects what symbolic admits
On a clean oracle the two checks usually agree. The geometric check matters where they *disagree* --- a question whose answer is symbolically constructible but whose declared fact triple is implausible. We forge the overdraft-fee question with its answer triple corrupted to the *wire* fee (\$45). The symbolic recompute still returns the true value, but the geometry refuses the implausible triple. This is the *same* `eval` generator family that mined everything above --- the guard, not a separate gadget.

**What the next cell does** --- forges a true and a corrupted question and validates each both ways:

1. **Read the true fact.** Find the overdraft ENM key and its scalar value; `answer_fn` returns it from the store.
2. **Build two questions.** `true_q` is faithful; `bad_q` reuses the same `answer_fn` (so it recomputes correctly) but declares an implausible `answer_triples` of the wire fee.
3. **Validate.** Run `_validate` for each on a symbolic-only and a geometric generator, and print the table --- both pass symbolic, only the geometry rejects `bad_q`.

In [ ]:
from knowlytix.benchmark.eval.generators.base import GeneratedQuestion

key = next(k for k in store.enm._store if 'overdraft' in k.id.lower())
val = float(store.enm._store[key].value.item())
answer_fn = lambda s, k=key: float(s.enm._store[k].value.item())

true_q = GeneratedQuestion('demo-true', 'exact_recall', 'What is the overdraft fee?',
                           answer_fn, val, '', 'float', metadata={'enm_id': key.id})
# symbolically valid (same answer_fn) but its DECLARED triple is the wire fee, not overdraft's
bad_q  = GeneratedQuestion('demo-bad', 'exact_recall', 'What is the overdraft fee?',
                           answer_fn, val, '', 'float',
                           metadata={'answer_triples': [('overdraft', 'has_fee_amount', '45.0')]})

g_sym, g_geo = ExactRecallGenerator(geometric=False), ExactRecallGenerator(geometric=True)
print(f"{'question':16s}{'symbolic':>12s}{'geometric':>12s}")
print(f"{'true fact':16s}{str(g_sym._validate(store, true_q)):>12s}{str(g_geo._validate(store, true_q)):>12s}")
print(f"{'implausible':16s}{str(g_sym._validate(store, bad_q)):>12s}{str(g_geo._validate(store, bad_q)):>12s}"
      '   <- geometry rejects what symbolic admits')

## Self-check
The asserts pin every number this notebook claims: the per-category admission counts and recovery splits, that nothing was silently rejected, that the clean oracle yields no contradictions or thresholds, that `multi_hop` carries a genuine two-hop `(head, value, tails)` ground truth, and that the geometry alone rejects the corrupted triple. Run it end to end and it fails loud if any of these drift.

In [ ]:
ex_g, ex_q = mined['exact_recall']
ct_g, ct_q = mined['counting']
xr_g, xr_q = mined['cross_reference']
mh_g, mh_q = mined['multi_hop']

assert len(ex_q) == 26 and ex_g.recovery_stats['two_way'] == 26            # all two-way
assert len(ct_q) == 25 and ct_g.recovery_stats['symbolic'] == 25 \
       and ct_g.recovery_stats['two_way'] == 0                             # aggregation -> symbolic
assert len(xr_q) == 4 and xr_g.recovery_stats['two_way'] == 4             # set-overlap two-way
assert len(mh_q) == 23 and mh_g.recovery_stats['two_way'] == 22 \
       and mh_g.recovery_stats['symbolic'] == 1                           # genuine two-hop
assert all(g.recovery_stats['rejected'] == 0 for g, _ in mined.values())  # nothing silently kept
assert len(mined['contradiction'][1]) == 0 and len(mined['threshold'][1]) == 0  # clean oracle
mh_two = next(q for q in mh_q if q.metadata.get('_recovery') == 'two_way')
assert isinstance(mh_two.ground_truth, tuple) and len(mh_two.ground_truth) == 3  # (head, value, tails)
assert mh_two.metadata.get('relation')                                    # second-hop relation present
assert g_sym._validate(store, true_q) and g_geo._validate(store, true_q)  # both admit truth
assert g_sym._validate(store, bad_q) and not g_geo._validate(store, bad_q)  # geometry alone rejects
print('OK - 5c: production generators, two-way geometric guard, genuine multi-hop')